### Setup

In [3]:
import os
import requests
from dotenv import load_dotenv
from kalshi_python_sync import Configuration, KalshiClient

# prod env
BASE_URL = "https://external-api.kalshi.com/trade-api/v2"
# demo env
# BASE_URL = "https://external-api.demo.kalshi.co/trade-api/v2"

### Exploring Events

In [4]:
response = requests.get(
    f"{BASE_URL}/events",
    params={
        "limit": 20,
        "status": "open",
        "with_nested_markets": True
    }
)
events = response.json().get("events", [])
print(f"Found {len(events)} events")

Found 20 events


In [5]:
for event in events[:5]:
    print(f"Event: {event['title']}")
    print(f"  Category: {event.get('category', 'N/A')}")
    print(f"  Series: {event.get('series_ticker', 'N/A')}")
    print(f"  Markets inside: {len(event.get('markets', []))}")
    print()

Event: Will Elon Musk visit Mars in his lifetime?
  Category: World
  Series: KXELONMARS
  Markets inside: 1

Event: Who will the next Pope be?
  Category: Elections
  Series: KXNEWPOPE
  Markets inside: 7

Event: Will the world pass 2 degrees Celsius over pre-industrial levels before 2050?
  Category: Climate and Weather
  Series: KXWARMING
  Markets inside: 1

Event: Will a human land on Mars before California starts high-speed rail?
  Category: Science and Technology
  Series: KXMARSVRAIL
  Markets inside: 1

Event: Will a supervolcano erupt before 2050?
  Category: Climate and Weather
  Series: KXERUPTSUPER
  Markets inside: 1



### Markets Inside an Event

In [6]:
event = events[1]
markets = event.get("markets", [])

print(f"Event: {event['title']}")
print(f"Category: {event.get('category', 'N/A')}")
print(f"Number of markets inside: {len(markets)}\n")

for m in markets:
    print(f"  {m.get('yes_sub_title', m['ticker'])}")
    print(f"    Ticker: {m['ticker']}")
    print(f"    Bid: {m.get('yes_bid_dollars', 'N/A')} / Ask: {m.get('yes_ask_dollars', 'N/A')}")
    print()

Event: Who will the next Pope be?
Category: Elections
Number of markets inside: 7

  Pierbattista Pizzaballa
    Ticker: KXNEWPOPE-70-PPIZ
    Bid: 0.0340 / Ask: 0.0440

  Pietro Parolin
    Ticker: KXNEWPOPE-70-PPAR
    Bid: 0.0460 / Ask: 0.0500

  Fridolin Ambongo
    Ticker: KXNEWPOPE-70-FAMB
    Bid: 0.0240 / Ask: 0.0330

  Luis Antonio Tagle
    Ticker: KXNEWPOPE-70-LANT
    Bid: 0.0400 / Ask: 0.0500

  Matteo Zuppi
    Ticker: KXNEWPOPE-70-MZUP
    Bid: 0.0320 / Ask: 0.0420

  Peter Erdo
    Ticker: KXNEWPOPE-70-PERD
    Bid: 0.0250 / Ask: 0.0350

  Anders Arborelius
    Ticker: KXNEWPOPE-70-AARB
    Bid: 0.0300 / Ask: 0.0390



### Browsing Markets Directly

In [7]:
response = requests.get(
    f"{BASE_URL}/markets",
    params={
        "limit": 20,
        "status": "open",
        #"mve_filter": "exclude",
    }
)
markets_list = response.json().get("markets", [])

for m in markets_list[:5]:
    print(f"Market: {m.get('yes_sub_title', m['ticker'])}")
    print(f"  Ticker: {m['ticker']}")
    print(f"  Event: {m.get('event_ticker', 'N/A')}")
    print(f"  Bid: {m.get('yes_bid_dollars', 'N/A')} / Ask: {m.get('yes_ask_dollars', 'N/A')}")
    print(f"  Volume 24h: {m.get('volume_24h_fp', 'N/A')}")
    print()

Market: yes Shai Gilgeous-Alexander: 1+,yes Stephon Castle: 1+,yes Chet Holmgren: 10+,yes Dylan Harper: 10+,yes Stephon Castle: 15+,yes Victor Wembanyama: 20+
  Ticker: KXMVESPORTSMULTIGAMEEXTENDED-S20269D00871CB29-EF70DE0848B
  Event: KXMVESPORTSMULTIGAMEEXTENDED-S20269D00871CB29
  Bid: 0.0000 / Ask: 0.2520
  Volume 24h: 0.00

Market: yes Shai Gilgeous-Alexander: 6+,yes De'Aaron Fox: 4+,yes Stephon Castle: 6+,yes Victor Wembanyama: 2+,yes San Antonio,yes De'Aaron Fox: 10+,yes Victor Wembanyama: 20+,yes Isaiah Hartenstein: 6+,yes De'Aaron Fox: 2+,yes Stephon Castle: 2+,yes Over 215.5 points scored
  Ticker: KXMVESPORTSMULTIGAMEEXTENDED-S20266097069E39E-9D71B203A65
  Event: KXMVESPORTSMULTIGAMEEXTENDED-S20266097069E39E
  Bid: 0.0000 / Ask: 0.3860
  Volume 24h: 0.00

Market: yes Braves wins by over 4.5 runs,yes Over 6.5 runs scored,yes Over 5.5 runs scored,yes Over 4.5 runs scored
  Ticker: KXMVECROSSCATEGORY-S2026335F55A96D6-5BA68B0682E
  Event: KXMVECROSSCATEGORY-S2026335F55A96D6
  Bid

### Finding a Market

In [8]:
# fetch all series, see what categories exist
resp = requests.get(f"{BASE_URL}/series")
all_series = resp.json().get("series", [])

categories = {}
for s in all_series:
    cat = s.get("category", "")
    if cat:
        categories.setdefault(cat, []).append(s)
print(f"Total series: {len(all_series)}")
print(f"\nCategories:")
for cat, items in sorted(categories.items()):
    print(f"  {cat}  ({len(items)} series)")

Total series: 10501

Categories:
  Climate and Weather  (274 series)
  Commodities  (53 series)
  Companies  (142 series)
  Crypto  (233 series)
  Economics  (546 series)
  Education  (1 series)
  Elections  (1366 series)
  Entertainment  (2423 series)
  Exotics  (10 series)
  Financials  (469 series)
  Health  (96 series)
  Mentions  (362 series)
  Politics  (1941 series)
  Science and Technology  (257 series)
  Social  (52 series)
  Sports  (2051 series)
  Transportation  (39 series)
  World  (143 series)


In [9]:
# pick a category, see what series are in it
crypto_series = categories["Crypto"]
print(f"Crypto series: {len(crypto_series)}\n")

for s in crypto_series[:10]:
    print(f"  {s['ticker']:30s} | {s['title']}")

Crypto series: 233

  KXXRPMAXMON                    | XRP Max Monthly
  KXSOLTXCOUNT                   | SOL transaction count
  KXDOGEMAXW                     | Dogecoin max
  KXDOGE15M                      | Dogecoin 15 Minute
  KXNEXTCOINCB                   | Next coin
  KXELSALVADORBTC                | El Salvador bitcoin
  KXETHMAXM                      | Ethereum max monthly
  KXSOL26500                     | Sol above 500 in 2026
  KXDOGEMAXM                     | Doge max monthly
  KXXRPATH                       | XRP ATH 


In [10]:
# pick a series, fetch its open events
resp = requests.get(
    f"{BASE_URL}/events",
    params={"series_ticker": "KXBTC15M",
            "status": "open",
            "with_nested_markets": True
            }
)
btc_events = resp.json().get("events", [])
print(f"BTC events: {len(btc_events)}\n")
for e in btc_events:
    markets = e.get("markets", [])
    print(f"Event: {e['title']}")
    print(f"  Ticker: {e['event_ticker']}")
    print(f"  Markets: {len(markets)}")
    for m in markets:
        print(f"    {m.get('yes_sub_title', m['ticker'])} - Bid: {m.get('yes_bid_dollars', 'N/A')} / Ask: {m.get('yes_ask_dollars', 'N/A')}")
    print()

BTC events: 1

Event: BTC 15 min · $73,203.99 target
  Ticker: KXBTC15M-26MAY300100
  Markets: 1
    Target Price: $73,203.99 - Bid: 0.8900 / Ask: 0.9000



In [12]:
# pick the market
event = btc_events[0]
markets = event.get("markets", [])
ticker = markets[0]["ticker"]

print(f"Event: {event['title']}")
print(f"Market: {markets[0].get('yes_sub_title', ticker)}")
print(f"Ticker: {ticker}")
print(f"Bid: {markets[0].get('yes_bid_dollars', 'N/A')} / Ask: {markets[0].get('yes_ask_dollars', 'N/A')}\n")

Event: BTC 15 min · $73,203.99 target
Market: Target Price: $73,203.99
Ticker: KXBTC15M-26MAY300100-00
Bid: 0.8900 / Ask: 0.9000

